# 04 · Series temporales financieras

**Financial Analytics Lab · Python · pandas · matplotlib**

**Pregunta orientadora:** ¿cómo cambia nuestra interpretación del rendimiento y del riesgo cuando estudiamos los activos a lo largo del tiempo?

En los módulos anteriores construimos un portafolio, preparamos precios históricos y estudiamos sus distribuciones. Ahora analizaremos el orden temporal de los datos: frecuencias, rezagos, rendimientos acumulados, medias móviles, volatilidad móvil y caídas desde máximos.

**Objetivo:** calcular indicadores verificables, comprender sus representaciones gráficas y elaborar conclusiones financieras que distingan la evidencia histórica de las predicciones.

**Modalidad:** lee las explicaciones, ejecuta el código, resuelve las prácticas y desarrolla el reto integrador. Se requiere conexión a internet para consultar la fuente externa.

## 0 · ¿Qué es una serie temporal?

Una serie temporal es una secuencia de observaciones asociadas a fechas o períodos. Su **orden** importa: el mismo conjunto de rendimientos, colocado en distinto orden, puede producir trayectorias y episodios de riesgo diferentes.

**Índice temporal:** identifica el momento de cada observación. **Frecuencia:** periodicidad (sesiones bursátiles, meses). **Rezago:** observación anterior. **Ventana móvil:** conjunto de observaciones recientes sobre el que se calcula una estadística.

**Reflexiona:** ¿por qué no debemos suponer que todos los días calendario tienen precios bursátiles?

## 1 · Consultar y validar los precios

Recuperaremos MSFT, AAPL y JPM, los mismos activos de los módulos anteriores. Puedes utilizar el archivo `precios_portafolio.csv` generado previamente o realizar una consulta nueva. El rango se registra con fechas explícitas; `end` es exclusivo y `auto_adjust=True` solicita ajustes según la fuente.

In [ ]:
%pip -q install yfinance pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timezone

TICKERS = ["MSFT", "AAPL", "JPM"]
INICIO = "2023-01-01"
FIN = "2025-01-01"
USAR_CSV = False
print("Consulta UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

In [ ]:
if USAR_CSV:
    from google.colab import files
    archivos = files.upload()
    if "precios_portafolio.csv" not in archivos:
        raise FileNotFoundError("Sube precios_portafolio.csv del módulo 02.")
    precios = pd.read_csv("precios_portafolio.csv", index_col="Fecha", parse_dates=["Fecha"])
else:
    series = []
    for ticker in TICKERS:
        tabla = yf.download(ticker, start=INICIO, end=FIN,
                            auto_adjust=True, progress=False, multi_level_index=False)
        if tabla.empty or "Close" not in tabla.columns:
            raise RuntimeError(f"No se obtuvieron cierres para {ticker}.")
        series.append(tabla[["Close"]].rename(columns={"Close": ticker}))
    precios = pd.concat(series, axis=1, join="inner")

precios.index = pd.to_datetime(precios.index)
precios = precios.sort_index()
precios = precios.loc[~precios.index.duplicated(keep="first")]
precios = precios.apply(pd.to_numeric, errors="coerce").dropna(how="any")
precios = precios.loc[(precios > 0).all(axis=1)]
if len(precios) < 60:
    raise ValueError("Se necesitan al menos 60 observaciones válidas.")
print("Dimensiones:", precios.shape)
display(precios.head())

**Interpretación:** cada fila representa una fecha común a los activos. Los días sin negociación no deben completarse con precios ficticios.

**Tu práctica 1:** verifica que el índice sea de fechas, esté ordenado y no contenga duplicados. Explica por qué trabajar con fechas comunes facilita las comparaciones.

In [ ]:
# TU PRÁCTICA 1
# Examina precios.index, orden y duplicados.

## 2 · Rezagos y rendimientos

`shift(1)` recupera el precio de la observación anterior. El rendimiento simple se define como **r(t) = P(t) / P(t−1) − 1**. `pct_change(fill_method=None)` aplica esa relación a toda la serie.

La observación anterior no siempre corresponde al día calendario anterior: entre sesiones bursátiles pueden existir fines de semana y festivos.

In [ ]:
activo = precios.columns[0]
verificacion = pd.DataFrame({
    "Precio": precios[activo],
    "Precio anterior": precios[activo].shift(1)
})
verificacion["Manual"] = verificacion["Precio"] / verificacion["Precio anterior"] - 1
verificacion["Pandas"] = precios[activo].pct_change(fill_method=None)
display(verificacion.head())
assert np.allclose(verificacion["Manual"].dropna(), verificacion["Pandas"].dropna())
rendimientos = precios.pct_change(fill_method=None).dropna(how="any")
print("Rendimientos contrastados.")

**Tu práctica 2:** verifica el rendimiento de otro activo con `shift(1)` y explica por qué el primer resultado es faltante.

In [ ]:
# TU PRÁCTICA 2
# Contrasta la fórmula en otro activo.

## 3 · Frecuencias y cierres mensuales

`resample('ME').last()` selecciona el último precio disponible de cada mes calendario. El rendimiento mensual se calcula comparando **cierres mensuales consecutivos**, no promediando rendimientos diarios. Cambiar la frecuencia modifica la cantidad de observaciones y la escala de interpretación.

In [ ]:
cierres_mensuales = precios.resample("ME").last().dropna(how="any")
rendimientos_mensuales = cierres_mensuales.pct_change(fill_method=None).dropna(how="any")
print("Observaciones originales:", len(precios))
print("Cierres mensuales:", len(cierres_mensuales))
display(cierres_mensuales.head())
display((rendimientos_mensuales * 100).head().round(3))

**Tu práctica 3:** comprueba manualmente un rendimiento mensual y explica por qué no equivale al promedio de los rendimientos diarios.

In [ ]:
# TU PRÁCTICA 3
# Verifica dos cierres mensuales consecutivos.

## 4 · Rendimiento acumulado

El rendimiento acumulado se obtiene multiplicando los factores **(1 + rendimiento)** de cada período y restando uno. En una serie de precios consistente y sin flujos adicionales también equivale a **precio actual / precio inicial − 1**.

**No se calcula sumando rendimientos simples.** El gráfico muestra cómo cambia el valor relativo desde el comienzo del período.

In [ ]:
acumulado = (1 + rendimientos).cumprod() - 1
acumulado_precios = precios.loc[rendimientos.index].div(precios.iloc[0]).sub(1)
assert np.allclose(acumulado.to_numpy(), acumulado_precios.to_numpy(), atol=1e-10)
ax = (acumulado * 100).plot(figsize=(10, 4))
ax.set(title="Rendimiento acumulado", xlabel="Fecha", ylabel="Rendimiento acumulado (%)")
ax.axhline(0, linewidth=1)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Tu práctica 4:** describe un período de caída o recuperación en la curva de un activo. Diferencia el rendimiento acumulado del rendimiento entre dos observaciones.

In [ ]:
# TU PRÁCTICA 4
# Selecciona un activo e interpreta su trayectoria.

## 5 · Medias móviles

Una media móvil de **n observaciones** calcula el promedio de los últimos n precios. Se utiliza para suavizar movimientos de corto plazo. En datos bursátiles, 20 observaciones no equivalen necesariamente a 20 días calendario.

`rolling(window=20, min_periods=20).mean()` requiere 20 precios antes de producir el primer valor. Las medias móviles describen trayectorias pasadas, no predicen precios.

In [ ]:
activo = precios.columns[0]
moviles = pd.DataFrame({"Precio": precios[activo]})
moviles["MM20"] = moviles["Precio"].rolling(20, min_periods=20).mean()
moviles["MM50"] = moviles["Precio"].rolling(50, min_periods=50).mean()
ax = moviles.plot(figsize=(10, 4))
ax.set(title=f"Precio y medias móviles: {activo}", xlabel="Fecha", ylabel="USD por acción")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Tu práctica 5:** incorpora una media móvil de 10 observaciones. ¿Cuál responde más rápido a las fluctuaciones, MM10 o MM50? ¿Por qué hay valores faltantes al inicio?

In [ ]:
# TU PRÁCTICA 5
# Calcula MM10 y compara ventanas.

## 6 · Volatilidad móvil

La volatilidad histórica móvil se aproxima mediante la **desviación estándar de los rendimientos** dentro de una ventana. `rolling(20).std(ddof=1)` calcula la dispersión muestral de 20 observaciones consecutivas.

Un aumento de la volatilidad indica mayor variabilidad local de rendimientos; no revela por sí solo si el activo subió o bajó. Presentaremos el resultado **en porcentaje por observación**, sin confundirlo con una medida anualizada.

In [ ]:
volatilidad_20 = rendimientos.rolling(20, min_periods=20).std(ddof=1) * 100
ax = volatilidad_20.plot(figsize=(10, 4))
ax.set(title="Volatilidad histórica móvil: 20 observaciones",
       xlabel="Fecha", ylabel="Desviación estándar (%) por observación")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Tu práctica 6:** compara ventanas de 10 y 30 observaciones. Identifica un episodio de variabilidad elevada y explica qué información adicional necesitarías para interpretarlo.

In [ ]:
# TU PRÁCTICA 6
# Calcula y compara volatilidades móviles.

## 7 · Drawdown: caídas desde máximos

El **drawdown** compara el valor de una inversión con su máximo acumulado previo: **drawdown(t) = valor(t) / máximo_hasta_t − 1**. Sus valores son cero o negativos.

El **máximo drawdown** es la caída más profunda observada durante el período. Este indicador describe la trayectoria de pérdidas desde máximos y complementa, pero no sustituye, la desviación estándar.

In [ ]:
indice = precios.div(precios.iloc[0])
drawdown = indice.div(indice.cummax()).sub(1)
max_drawdown = drawdown.min() * 100
display(max_drawdown.to_frame("Máximo drawdown (%)").round(2))
ax = (drawdown * 100).plot(figsize=(10, 4))
ax.set(title="Caídas desde máximos previos", xlabel="Fecha", ylabel="Drawdown (%)")
ax.axhline(0, linewidth=1)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
assert (drawdown <= 1e-10).all().all()

**Tu práctica 7:** identifica el activo con el drawdown más profundo. Explica por qué no necesariamente coincide con el de mayor desviación estándar.

In [ ]:
# TU PRÁCTICA 7
# Compara el drawdown y la dispersión.

## 8 · Síntesis: comparar sin confundir indicadores

El rendimiento acumulado describe el cambio de valor; las medias móviles suavizan la trayectoria; la volatilidad móvil representa dispersión reciente; el drawdown muestra caídas respecto de máximos previos.

**Interpretación rigurosa:** no existe un indicador único que determine la mejor inversión. Las conclusiones deben señalar período, fuente, supuestos y limitaciones.

In [ ]:
resumen = pd.DataFrame({
    "Rendimiento acumulado (%)": acumulado.iloc[-1] * 100,
    "Volatilidad por observación (%)": rendimientos.std(ddof=1) * 100,
    "Máximo drawdown (%)": max_drawdown,
    "Precio inicial": precios.iloc[0],
    "Precio final": precios.iloc[-1]
})
display(resumen.round(3))
resumen.to_csv("resumen_series_temporales.csv", index_label="Ticker")
print("Exportado: resumen_series_temporales.csv")

**Tu práctica 8:** compara dos activos mediante tres indicadores y redacta una conclusión de 120 a 160 palabras que distinga resultados históricos de expectativas futuras.

## 9 · Reto integrador: informe de series temporales

Elabora un análisis original de **tres activos** con:

1. Pregunta financiera, tickers, fuente y período.
2. Validación del índice temporal y de fechas comunes.
3. Rendimientos simples con comprobación manual.
4. Cierres mensuales y verificación de un rendimiento mensual.
5. Rendimiento acumulado y gráfico interpretado.
6. Medias móviles de 20 y 50 observaciones.
7. Volatilidad móvil de 20 observaciones y comparación temporal.
8. Drawdown y máximo drawdown.
9. Tabla de indicadores exportada a CSV.
10. Conclusión de **200 a 300 palabras**, con hallazgos, supuestos y limitaciones.

**Evidencia:** notebook reproducible con explicaciones, cálculos verificados, gráficos rotulados y fuentes. Reconoce cualquier apoyo de herramientas de IA.

In [ ]:
# RETO INTEGRADOR — TU SOLUCIÓN
# Desarrolla aquí tus cálculos, gráficos e interpretación.

## 10 · Autoevaluación y continuidad

- [ ] Comprendo el índice temporal y las frecuencias.
- [ ] Calculo rezagos y rendimientos correctamente.
- [ ] Distingo rendimientos mensuales de diarios.
- [ ] Interpreto el rendimiento acumulado.
- [ ] Comprendo medias y volatilidades móviles.
- [ ] Calculo e interpreto drawdown.
- [ ] Presento conclusiones con límites explícitos.

**Próximo módulo:** relaciones entre activos, correlación, covarianza, regresión y simulación de escenarios.

**Referencias:** [pandas: series temporales](https://pandas.pydata.org/docs/user_guide/timeseries.html) · [pandas: ventanas](https://pandas.pydata.org/docs/user_guide/window.html) · [yfinance](https://ranaroussi.github.io/yfinance/) · [matplotlib](https://matplotlib.org/stable/).

Los resultados históricos no son recomendaciones de inversión.